<a href="https://colab.research.google.com/github/Goseungeun/2022Master/blob/main/Part2/Chapter4_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 문제 정의

- 아울렛 매장 1,500여개 제품에 대한 판매 데이터 활용 -> 예측 모델을 만들과 각 제품의 판매금액 예측
- 평가 기준 : RMSE
- label(target) : 판매금액 (Item_Outlet_Sale)
- 제출파일 : 예측값만 result.csv 파일로 생성해 제출 (컬럼명 : pred, 1개)

In [1]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch4/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch4/test.csv")

## EDA

In [2]:
train.shape , test.shape

((6818, 12), (1705, 11))

In [5]:
train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,NCR06,12.500,Low Fat,0.006760,Household,42.8112,OUT013,1987,High,Tier 3,Supermarket Type1,639.1680
1,FDW11,12.600,Low Fat,0.048741,Breads,60.4194,OUT013,1987,High,Tier 3,Supermarket Type1,990.7104
2,FDH32,12.800,Low Fat,0.075997,Fruits and Vegetables,97.1410,OUT013,1987,High,Tier 3,Supermarket Type1,2799.6890
3,FDL52,6.635,Regular,0.046351,Frozen Foods,37.4506,OUT017,2007,NaN,Tier 2,Supermarket Type1,1176.4686
4,FDO09,13.500,Regular,0.125170,Snack Foods,261.4910,OUT013,1987,High,Tier 3,Supermarket Type1,3418.8830


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            6818 non-null   object 
 1   Item_Weight                5656 non-null   float64
 2   Item_Fat_Content           6818 non-null   object 
 3   Item_Visibility            6818 non-null   float64
 4   Item_Type                  6818 non-null   object 
 5   Item_MRP                   6818 non-null   float64
 6   Outlet_Identifier          6818 non-null   object 
 7   Outlet_Establishment_Year  6818 non-null   int64  
 8   Outlet_Size                4878 non-null   object 
 9   Outlet_Location_Type       6818 non-null   object 
 10  Outlet_Type                6818 non-null   object 
 11  Item_Outlet_Sales          6818 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 639.3+ KB


In [7]:
train.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
count,5656.000000,6818.000000,6818.000000,6818.000000,6818.000000
mean,12.872703,0.066121,140.419533,1997.885890,2190.941459
std,4.651034,0.051383,62.067861,8.339795,1706.131256
min,4.555000,0.000000,31.290000,1985.000000,33.290000
25%,8.785000,0.026914,93.610050,1987.000000,836.577700
50%,12.600000,0.053799,142.448300,1999.000000,1806.648300
75%,17.000000,0.095273,185.060150,2004.000000,3115.944000
max,21.350000,0.328391,266.888400,2009.000000,13086.964800


In [8]:
train.describe(include='O')

,Item_Identifier,Item_Fat_Content,Item_Type,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type
count,6818,6818,6818,6818,4878,6818,6818
unique,1554,5,16,10,3,3,4
top,FDO19,Low Fat,Snack Foods,OUT046,Medium,Tier 3,Supermarket Type1
freq,9,4092,963,763,2228,2664,4474


In [9]:
test.describe(include='O')

,Item_Identifier,Item_Fat_Content,Item_Type,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type
count,1705,1705,1705,1705,1235,1705,1705
unique,1077,5,16,10,3,3,4
top,DRD15,Low Fat,Fruits and Vegetables,OUT013,Medium,Tier 3,Supermarket Type1
freq,4,997,272,207,565,686,1103


In [10]:
train.isnull().sum()

,0
Item_Identifier,0
Item_Weight,1162
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,1940
Outlet_Location_Type,0


In [11]:
test.isnull().sum()

,0
Item_Identifier,0
Item_Weight,301
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,470
Outlet_Location_Type,0


## Data Preprocessing

In [12]:
list(train.columns[train.dtypes == object])

['Item_Identifier',
 'Item_Fat_Content',
 'Item_Type',
 'Outlet_Identifier',
 'Outlet_Size',
 'Outlet_Location_Type',
 'Outlet_Type']

In [13]:
cols = [
 'Item_Fat_Content',
 'Item_Type',
 'Outlet_Identifier',
 'Outlet_Size',
 'Outlet_Location_Type',
 'Outlet_Type']

In [14]:
target = train.pop('Item_Outlet_Sales')
print(train.shape,test.shape)
df = pd.concat([train,test])
print(df.shape)

(6818, 11) (1705, 11)
(8523, 11)


In [16]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in cols:
  df[col] = le.fit_transform(df[col])

In [17]:
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,NCR06,12.500,1,0.006760,9,42.8112,1,1987,0,2,1
1,FDW11,12.600,1,0.048741,1,60.4194,1,1987,0,2,1
2,FDH32,12.800,1,0.075997,6,97.1410,1,1987,0,2,1
3,FDL52,6.635,2,0.046351,5,37.4506,2,2007,3,1,1
4,FDO09,13.500,2,0.125170,13,261.4910,1,1987,0,2,1


In [18]:
train = df.iloc[:len(train)].copy()
test = df.iloc[len(train):].copy()
train.shape , test.shape

((6818, 11), (1705, 11))

In [19]:
train['Item_Weight'] = train['Item_Weight'].fillna(train['Item_Weight'].min())
train['Outlet_Size'] = train['Outlet_Size'].fillna(train['Outlet_Size'].mode()[0])

test['Item_Weight'] = train['Item_Weight'].fillna(train['Item_Weight'].min())
test['Outlet_Size'] = train['Outlet_Size'].fillna(train['Outlet_Size'].mode()[0])

In [21]:
print(train.shape,test.shape)
train.drop('Item_Identifier',axis=1,inplace=True)
test.drop('Item_Identifier',axis=1,inplace=True)
print(train.shape,test.shape)

(6818, 10) (1705, 10)


KeyError: "['Item_Identifier'] not found in axis"

## Split Data

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val = train_test_split(train,target,test_size = 0.2,random_state = 0)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((5454, 10), (1364, 10), (5454,), (1364,))

## Machine Learning Train & validation

In [24]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error,mean_absolute_error

In [26]:
#선형 회귀
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred = lr.predict(X_val)

result = mean_squared_error(y_val,y_pred)
print("MSE :",result)

result = mean_absolute_error(y_val,y_pred)
print("MAE :",result)

result = root_mean_squared_error(y_val,y_pred)
print("RMSE :",result)

result = r2_score(y_val,y_pred)
print("R2 :",result)

MSE : 1282923.072983389
MAE : 865.1968401416275
RMSE : 1132.6619411737065
R2 : 0.5058168396924844


In [27]:
# 랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_val)

result = mean_squared_error(y_val,y_pred)
print("MSE :",result)

result = mean_absolute_error(y_val,y_pred)
print("MAE :",result)

result = root_mean_squared_error(y_val,y_pred)
print("RMSE :",result)

result = r2_score(y_val,y_pred)
print("R2 :",result)

MSE : 1126925.8302996708
MAE : 751.369494621701
RMSE : 1061.567628698083
R2 : 0.5659071225879551


In [28]:
#lightGBM
import lightgbm as lgb
model = lgb.LGBMRegressor(random_state = 0,verbose=-1)
model.fit(X_train,y_train)
y_pred = model.predict(X_val)

result = mean_squared_error(y_val,y_pred)
print("MSE :",result)

result = mean_absolute_error(y_val,y_pred)
print("MAE :",result)

result = root_mean_squared_error(y_val,y_pred)
print("RMSE :",result)

result = r2_score(y_val,y_pred)
print("R2 :",result)

MSE : 1115654.3482227568
MAE : 736.6367966578568
RMSE : 1056.2454015155554
R2 : 0.5702489079618556


## Predict & Create Result CSV

In [29]:
pred = model.predict(test)
pred

array([1268.1235854 ,  829.4139342 , 1890.59805142, ..., 3171.60340394,
       1366.19341927, 1275.12373822])

In [30]:
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv",index=False)

In [31]:
pd.read_csv("result.csv")

,pred
0,1268.123585
1,829.413934
2,1890.598051
3,1662.497076
4,2752.922146
...,...
1700,285.648607
1701,536.671136
1702,3171.603404
1703,1366.193419
